In [0]:
import uuid
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime

# Generate unique identifiers for this ETL run
SSIDGUID = str(uuid.uuid4())
ETLRunTime = datetime.now()
IncrementalExtractTime = datetime.now()

# Configuration variables matching the original stored procedure
MasterSourceSystemName = ('PowerApps')
SourceSubSystemName = ('')
SourceDatabaseName = ('PowerApps')
SourceTableName = ('PowerApps')
DestinationSchemaName = ('NUE')
DestinationTableName = ('factCargoAllocationPowerApps')
DestinationDatabaseName = ('DJJStarSchema')
ETLType = ('Incremental')
BatchSize = 1000
PkgName = ('NUE.etl_factCargoAllocationPowerAppsCarryForward')
ETLTemplate = ('')
PkgGUID = str(uuid.uuid4())
DJJLastUpdateTime = datetime.now()

# Catalog mappings
# source_catalog: bronze layer for operational source systems
source_catalog = "bronze_non_prod"
# target_catalog: enriched/temporary layer for temp tables and target objects
target_catalog = "djj_enriched_non_prod"
# federated_starschema_catalog: star schema layer (DJJStarSchema -> djj_starschema)
federated_starschema_catalog = "djj_starschema"

# Audit counters
Success = '0'
RowsInserted = 0
RowsUpdated = 0
RowsDeleted = 0
ErrorRowCount = 0
TableInitialRowCount = 0
TableFinalRowCount = 0
ETLStatus = ('')
ExtractRowCount = 0
SSISGUID = SSIDGUID

### Execute the Initial Metadata Logging

In [0]:
%run /Workspace/Shared/metadata_framework/metadata_logger

In [0]:
#Insert/Update records in ETL Master
logger = ETLLogger(spark)

ETLID = logger.log_etl(
    MasterSourceSystemName=MasterSourceSystemName,
    SourceSubSystemName=SourceSubSystemName,
    SourceDatabaseName=SourceDatabaseName,
    SourceTableName=SourceTableName,
    DestinationDatabaseName=DestinationDatabaseName,
    DestinationTableName=DestinationTableName,
    PkgName=PkgName,
    PkgGUID=PkgGUID,
    ETLTemplate=ETLTemplate,
    ETLType=ETLType,
    CheckpointsEnabled=True
)

print(ETLID)

In [0]:
#Insert records in ETLExecutionLog table
exec_logger = ETLExecutionLogger(spark)
exec_logger.log_execution(
    ETLID=ETLID,
    ETLRunTime=ETLRunTime,
    Success=Success,
    RowsInserted=RowsInserted,
    RowsUpdated=RowsUpdated,
    ErrorRowCount=ErrorRowCount,
    TableInitialRowCount=TableInitialRowCount,
    TableFinalRowCount=TableFinalRowCount,
    ETLStatus=ETLStatus,
    ExtractRowCount=ExtractRowCount,
    SSISGUID=SSIDGUID
)

In [0]:
ETLExecutionID = spark.sql(f"""SELECT ETLExecutionID from {target_catalog}.metadata.etlexecutionlog where lower(trim(ETLID)) = lower(trim('{ETLID}')) and lower(trim(SSISGUID)) = lower(trim('{SSIDGUID}'))""").first()[0]

### ETL Logic Starts Here

In [0]:
# Get initial row count of target table
TableInitialRowCount = spark.sql(f"SELECT COUNT(1) AS cnt FROM {target_catalog}.NUE.factCargoAllocationPowerApps").first()["cnt"]
print(f"Initial row count: {TableInitialRowCount}")

In [0]:
# Get TargetLastUpdatedDate for incremental logic
target_last_updated_row = spark.sql(f"""
    SELECT COALESCE(MAX(CAST(EnrichedTimestampUTC AS DATE)), CAST('1900-01-01' AS DATE)) AS TargetLastUpdatedDate
    FROM {target_catalog}.NUE.factCargoAllocationPowerApps
""").first()
TargetLastUpdatedDate = str(target_last_updated_row["TargetLastUpdatedDate"])
print(f"TargetLastUpdatedDate: {TargetLastUpdatedDate}")

In [0]:
# Source Query - Carry Forward Records
# Runs once a week, purpose is to carry any already created records forward to the next week
# as long as they have not become allocated.
# Needs to run after the daily ETL so it can accurately use the Historic Flag in this etl

# Create temp table with carry forward records
# Source tables: DJJStarSchema.Nue.factCargoAllocationPowerApps -> {target_catalog}.NUE.factCargoAllocationPowerApps
# Source tables: DJJStarSchema.DJJ.dimDate -> {federated_starschema_catalog}.DJJ.dimDate
# Source tables: DJJStarSchema.Nue.factCargoMillDetails -> {federated_starschema_catalog}.NUE.factCargoMillDetails
# Source tables: DJJStarSchema.Nue.dimNucorMillLocations -> {federated_starschema_catalog}.NUE.dimNucorMillLocations
# Source tables: DJJStarSchema.Brk.dimGrade -> {federated_starschema_catalog}.Brk.dimGrade
# Source tables: DJJStarSchema.nue.dimPorts -> {federated_starschema_catalog}.NUE.dimPorts
# Source tables: DJJStarSchema.Brk.factCargoPurchaseOrder -> {federated_starschema_catalog}.Brk.factCargoPurchaseOrder
# Source tables: DJJStarSchema.Brk.dimSupplier -> {federated_starschema_catalog}.Brk.dimSupplier
# Source tables: DJJStarSchema.Brk.dimCargo -> {federated_starschema_catalog}.Brk.dimCargo
# Note: dbo.fn_DateTimeToDateKey and dbo.fn_DateKeyToDateTime are SQL Server functions called as-is

spark.sql(f"""
    CREATE TABLE {target_catalog}.temp.factCargoAllocationPowerApps_CarryForward AS
    SELECT
        CASE WHEN CAST(RIGHT(CAST(a.FiscalWeekKey AS STRING), 2) AS INT) = 52
            THEN CAST(CONCAT(CAST(CAST(LEFT(CAST(a.FiscalWeekKey AS STRING), 4) AS INT) + 1 AS STRING), '01') AS INT)
            ELSE a.FiscalWeekKey + 1
        END AS FiscalWeekKey,
        a.CargoID,
        a.ShippingScenarioID,
        a.MillName,
        a.EnterpriseGrade,
        a.OriginalGTs,
        a.AllocatedGTs,
        a.CostPerGT AS CostPerGT,
        COALESCE(c.LoadStartDate, a.LoadStartDate) AS LoadStartDate,
        COALESCE(c.SailDate, a.SailDate) AS SailDate,
        COALESCE(c.TranDisPortETADate, a.TranDisPortETADate) AS TranDisPortETADate,
        COALESCE(c.DischargeFinishDate, a.DischargeFinishDate) AS DischargeFinishDate,
        COALESCE(c.MillTicketInitialArrivalDate, a.MillTicketInitialArrivalDate) AS MillTicketInitialArrivalDate,
        COALESCE(c.MeltEndDate, a.MeltEndDate) AS MeltEndDate,
        COALESCE(COALESCE(c.SupplierName, a.SupplierName), 'Unknown') AS SupplierName,
        COALESCE(COALESCE(c.OriginCountry, a.OriginCountry), 'Unknown') AS OriginCountry,
        COALESCE(COALESCE(c.DestinationPort, a.DestinationPort), 'Unknown') AS DestinationPort,
        COALESCE(COALESCE(c.Vessel, a.Vessel), 'TBD') AS Vessel,
        COALESCE(c.StatusDesc, a.StatusDesc) AS StatusDesc,
        a.HistoricFlag,
        a.LastUpdateTime,
        a.LastUpdateUser
    FROM {target_catalog}.NUE.factCargoAllocationPowerApps a
        INNER JOIN (
            SELECT MAX(a2.FiscalWeekKey) AS FiscalWeek
            FROM {target_catalog}.NUE.factCargoAllocationPowerApps a2
                INNER JOIN (
                    SELECT
                        Fiscal_Week_Of_Year
                    FROM {federated_starschema_catalog}.DJJ.dimDate
                    WHERE DateKey = dbo.fn_DateTimeToDateKey(DATE_SUB(current_date(), 14))
                ) b2
                    ON a2.FiscalWeekKey = b2.Fiscal_Week_Of_Year
            WHERE lower(trim(a2.MillName)) <> lower(trim('NUCOR UNALLOCATED'))
        ) maxweek
            ON a.FiscalWeekKey = maxweek.FiscalWeek

        INNER JOIN (
            SELECT
                MillName,
                FiscalWeekKey,
                CargoID,
                ShippingScenarioID,
                EnterpriseGrade
            FROM {target_catalog}.NUE.factCargoAllocationPowerApps a3
            WHERE a3.HistoricFlag = 0
                AND lower(trim(a3.MillName)) <> lower(trim('Nucor Unallocated'))
        ) b
            ON lower(trim(a.FiscalWeekKey)) = lower(trim(b.FiscalWeekKey))
            AND lower(trim(a.CargoID)) = lower(trim(b.CargoID))
            AND lower(trim(a.ShippingScenarioID)) = lower(trim(b.ShippingScenarioID))
            AND lower(trim(a.EnterpriseGrade)) = lower(trim(b.EnterpriseGrade))
            AND lower(trim(a.MillName)) = lower(trim(b.MillName))

        LEFT OUTER JOIN (
            SELECT
                d.Fiscal_Week_of_Year AS FiscalWeekKey,
                cmd.CargoID,
                cmd.ShippingScenarioID,
                nml.DJJLocationName AS MillName,
                dg.EnterpriseGradeGroup,
                CAST(cmd.OrderedWgt / 2240 AS INT) AS OriginalGTs,
                0 AS AllocatedGTs,
                CASE WHEN cmd.OrderedWgt / 2240 = 0 THEN 0 ELSE cmd.OrderedAmount / (cmd.OrderedWgt / 2240) END AS CostPerGT,
                CAST(dbo.fn_DateKeyToDateTime(
                    CASE WHEN cmd.LoadStartDateKey = 19000101
                        THEN cmd.ESTLoadStartDateKey
                        ELSE cmd.LoadStartDateKey
                    END) AS DATE) AS LoadStartDate,
                CAST(dbo.fn_DateKeyToDateTime(cmd.ESTSailDateKey) AS DATE) AS SailDate,
                CAST(dbo.fn_DateKeyToDateTime(cmd.ESTTranDisArrivalDateKey) AS DATE) AS TranDisPortETADate,
                CAST(dbo.fn_DateKeyToDateTime(cmd.ESTDischargeFinishDateKey) AS DATE) AS DischargeFinishDate,
                CAST(dbo.fn_DateKeyToDateTime(cmd.ESTMINMillTicketDateKey) AS DATE) AS MillTicketInitialArrivalDate,
                COALESCE(CAST(dbo.fn_DateKeyToDateTime(cmd.MeltEndDate) AS DATE), CAST('1900-01-01' AS DATE)) AS MeltEndDate,
                COALESCE(h.SupplierName, 'Unknown') AS SupplierName,
                COALESCE(e.PortCountryName, 'Unknown') AS OriginCountry,
                COALESCE(ed.PortName, 'Unknown') AS DestinationPort,
                COALESCE(f.VehicleName, 'TBD') AS Vessel,
                cmd.StatusDesc,
                0 AS HistoricFlag,
                current_timestamp() AS LastUpdateTime,
                'SQL' AS LastUpdateUser
            FROM {federated_starschema_catalog}.NUE.factCargoMillDetails cmd
                LEFT OUTER JOIN {federated_starschema_catalog}.NUE.dimNucorMillLocations nml
                    ON lower(trim(cmd.NucorLocationKey)) = lower(trim(nml.NucorLocationKey))
                LEFT OUTER JOIN {federated_starschema_catalog}.Brk.dimGrade dg
                    ON lower(trim(cmd.BrokerageGradeKey)) = lower(trim(dg.GradeKey))
                LEFT OUTER JOIN {federated_starschema_catalog}.DJJ.dimDate d
                    ON d.DateKey = dbo.fn_DateTimeToDateKey(DATE_SUB(current_date(), 7))
                LEFT OUTER JOIN {federated_starschema_catalog}.NUE.dimPorts e
                    ON lower(trim(cmd.OriginPortKey)) = lower(trim(e.PortKey))
                LEFT OUTER JOIN {federated_starschema_catalog}.NUE.dimPorts ed
                    ON lower(trim(cmd.DestinationPortKey)) = lower(trim(ed.PortKey))
                LEFT OUTER JOIN {federated_starschema_catalog}.Brk.factCargoPurchaseOrder g
                    ON lower(trim(cmd.ShippingScenarioID)) = lower(trim(g.ShippingScenarioID))
                LEFT OUTER JOIN {federated_starschema_catalog}.Brk.dimSupplier h
                    ON lower(trim(g.SupplierKey)) = lower(trim(h.SupplierKey))
                LEFT OUTER JOIN {federated_starschema_catalog}.Brk.dimCargo f
                    ON lower(trim(cmd.CargoKey)) = lower(trim(f.CargoKey))
            WHERE lower(trim(nml.DJJLocationName)) = lower(trim('NUCOR UNALLOCATED'))
                AND CAST(dbo.fn_DateKeyToDateTime(cmd.ExpirationDateKey) AS TIMESTAMP) > current_timestamp()
        ) c
            ON lower(trim(a.CargoID)) = lower(trim(c.CargoID))
            AND lower(trim(a.ShippingScenarioID)) = lower(trim(c.ShippingScenarioID))
""")
print("Temp table factCargoAllocationPowerApps_CarryForward created and populated.")

In [0]:
# Perform Updates - MERGE INTO target using carry forward temp table
# Original SQL uses UPDATE ... FROM with INNER JOIN on composite key
spark.sql(f"""
    MERGE INTO {target_catalog}.NUE.factCargoAllocationPowerApps a
    USING {target_catalog}.temp.factCargoAllocationPowerApps_CarryForward b
    ON lower(trim(a.FiscalWeekKey)) = lower(trim(b.FiscalWeekKey))
        AND lower(trim(a.CargoID)) = lower(trim(b.CargoID))
        AND lower(trim(a.ShippingScenarioID)) = lower(trim(b.ShippingScenarioID))
        AND lower(trim(a.EnterpriseGrade)) = lower(trim(b.EnterpriseGrade))
        AND lower(trim(a.MillName)) = lower(trim(b.MillName))
    WHEN MATCHED THEN UPDATE SET
        a.OriginalGTs = b.OriginalGTs,
        a.AllocatedGTs = b.AllocatedGTs,
        a.CostPerGT = b.CostPerGT,
        a.LoadStartDate = b.LoadStartDate,
        a.SailDate = b.SailDate,
        a.TranDisPortETADate = b.TranDisPortETADate,
        a.DischargeFinishDate = b.DischargeFinishDate,
        a.MillTicketInitialArrivalDate = b.MillTicketInitialArrivalDate,
        a.MeltEndDate = b.MeltEndDate,
        a.SupplierName = b.SupplierName,
        a.OriginCountry = b.OriginCountry,
        a.DestinationPort = b.DestinationPort,
        a.Vessel = b.Vessel,
        a.StatusDesc = b.StatusDesc,
        a.HistoricFlag = b.HistoricFlag,
        a.LastUpdateTime = b.LastUpdateTime,
        a.LastUpdateUser = b.LastUpdateUser
""")
print("Updates completed.")

In [0]:
# Get RowsUpdated count from the MERGE operation using Delta table history
history_df = spark.sql(f"DESCRIBE HISTORY {target_catalog}.NUE.factCargoAllocationPowerApps LIMIT 1")
operation_metrics = history_df.select("operationMetrics").first()[0]
RowsUpdated = int(operation_metrics.get("numTargetRowsUpdated", 0)) if operation_metrics else 0
print(f"Rows updated: {RowsUpdated}")

In [0]:
# Perform Inserts - Insert new carry forward records that do not already exist in the target
# Original SQL uses LEFT OUTER JOIN ... WHERE b.CargoID IS NULL pattern
spark.sql(f"""
    INSERT INTO {target_catalog}.NUE.factCargoAllocationPowerApps
    SELECT
        a.FiscalWeekKey,
        a.CargoID,
        a.ShippingScenarioID,
        a.MillName,
        a.EnterpriseGrade,
        a.OriginalGTs,
        a.AllocatedGTs,
        a.CostPerGT,
        a.LoadStartDate,
        a.SailDate,
        a.TranDisPortETADate,
        a.DischargeFinishDate,
        a.MillTicketInitialArrivalDate,
        a.MeltEndDate,
        a.SupplierName,
        a.OriginCountry,
        a.DestinationPort,
        a.Vessel,
        a.StatusDesc,
        a.HistoricFlag,
        a.LastUpdateTime,
        a.LastUpdateUser
    FROM {target_catalog}.temp.factCargoAllocationPowerApps_CarryForward a
        LEFT OUTER JOIN {target_catalog}.NUE.factCargoAllocationPowerApps b
            ON lower(trim(a.FiscalWeekKey)) = lower(trim(b.FiscalWeekKey))
            AND lower(trim(a.CargoID)) = lower(trim(b.CargoID))
            AND lower(trim(a.ShippingScenarioID)) = lower(trim(b.ShippingScenarioID))
            AND lower(trim(a.EnterpriseGrade)) = lower(trim(b.EnterpriseGrade))
            AND lower(trim(a.MillName)) = lower(trim(b.MillName))
    WHERE b.CargoID IS NULL
""")
print("Inserts completed.")

In [0]:
# Get RowsInserted count from the INSERT operation using Delta table history
history_df = spark.sql(f"DESCRIBE HISTORY {target_catalog}.NUE.factCargoAllocationPowerApps LIMIT 1")
operation_metrics = history_df.select("operationMetrics").first()[0]
RowsInserted = int(operation_metrics.get("numOutputRows", 0)) if operation_metrics else 0
print(f"Rows inserted: {RowsInserted}")

In [0]:
# Deletes - no deletes performed in this ETL
RowsDeleted = 0
print(f"Rows deleted: {RowsDeleted}")

### Close Out the ETL - Final Audit & Logging

In [0]:
# Get final rowcount of target table
final_count_row = spark.sql(f"""SELECT COUNT(1) AS cnt FROM {target_catalog}.NUE.factCargoAllocationPowerApps""").first()
TableFinalRowCount = final_count_row['cnt'] if final_count_row else 0
print(f"Final row count: {TableFinalRowCount}")
print(f"Rows added: {TableFinalRowCount - TableInitialRowCount}")

In [0]:
# Update records in ETLExecutionLog table with success status
exec_logger.log_execution(
    ETLID=ETLID,
    ETLRunTime=ETLRunTime,
    Success='1',
    RowsInserted=RowsInserted,
    RowsUpdated=RowsUpdated,
    ErrorRowCount=ErrorRowCount,
    TableInitialRowCount=TableInitialRowCount,
    TableFinalRowCount=TableFinalRowCount,
    ETLStatus=ETLStatus,
    ExtractRowCount=ExtractRowCount,
    SSISGUID=SSIDGUID
)

In [0]:
# Update ETLMaster with final metadata
spark.sql(f"""UPDATE {target_catalog}.metadata.ETLMaster
    SET lower(trim(IncrementalExtractTime)) = lower(trim('{IncrementalExtractTime}')), 
        lower(trim(ETLTemplate)) = lower(trim('{ETLTemplate}')), 
        lower(trim(ETLType)) = lower(trim('{ETLType}'))
    WHERE lower(trim(ETLID)) = lower(trim('{ETLID}'))""")

In [0]:
# Cleanup - drop temp table
spark.sql(f"DROP TABLE IF EXISTS {target_catalog}.temp.factCargoAllocationPowerApps_CarryForward")
print("Temp table cleaned up. ETL completed successfully.")